# RQ2 - Cross-architecture SK-ESD ranking (Coffee Lake vs Arrow Lake)

Cells are **verbatim** from `twin_skesd.ipynb`. The author's superseded duplicate of the
SK-ESD call (original cell 2, kept 'just in case') is not included; cell 3 is the version
that produced the published table.

**Inputs:** `./inputs/skesd_sample_coffee.pkl`, `./inputs/skesd_sample_pc2.pkl`

**Paper artifacts:** Fig. `fig:skesd` (`twinskesdplot.pdf`), Table `tableskesd`.

**Requires R** with `ScottKnottESD` + `ggplot2`, and `rpy2`.


## Data preparation: load the per-CPU SK-ESD sample matrices


In [1]:
import pickle as pkl
import matplotlib as plt
import numpy as np
import pandas as pd 


with open('./inputs/skesd_sample_coffee.pkl', "rb") as file: 
    skesd_sample_coffee = pkl.load(file)
with open('./inputs/skesd_sample_pc2.pkl', "rb") as file: 
    skesd_sample_pc2 = pkl.load(file)

print(type(skesd_sample_coffee))
print(skesd_sample_coffee.shape)
print(type(skesd_sample_pc2))
print(skesd_sample_pc2.shape)
print(skesd_sample_pc2.columns)
print(skesd_sample_pc2.head())

skesd_sample_pc2_trimmed = skesd_sample_pc2.head(101)
skesd_sample_coffee = skesd_sample_coffee.add_suffix("CoffeeLake")
skesd_sample_pc2_trimmed = skesd_sample_pc2_trimmed.add_suffix("ArrowLake")
skesd_sample = pd.concat([skesd_sample_coffee, skesd_sample_pc2_trimmed], axis=1)
print(skesd_sample.head())

<class 'pandas.core.frame.DataFrame'>
(101, 10)
<class 'pandas.core.frame.DataFrame'>
(404, 10)
Index(['fp64', 'int128', 'int32', 'int64', 'AVX2', 'fp32', 'SSE2', 'SSE',
       'MMX', 'AVX'],
      dtype='object')
       fp64    int128     int32     int64      AVX2      fp32      SSE2  \
0 -0.095365 -0.138294 -0.138968 -0.133545 -0.125065 -0.112552 -0.124613   
1 -0.091151 -0.128004 -0.123597 -0.134936 -0.112364 -0.100342 -0.120361   
2 -0.098237 -0.126862 -0.118434 -0.132750 -0.115440 -0.101880 -0.118246   
3 -0.089102 -0.128326 -0.127810 -0.129132 -0.109076 -0.109283 -0.115780   
4 -0.085796 -0.133564 -0.115901 -0.136557 -0.114070 -0.098940 -0.131741   

        SSE       MMX       AVX  
0 -0.116688 -0.112322 -0.126530  
1 -0.105864 -0.107684 -0.119557  
2 -0.116037 -0.106423 -0.116540  
3 -0.105622 -0.109178 -0.116935  
4 -0.113432 -0.107231 -0.115119  
   int32CoffeeLake  int64CoffeeLake  int128CoffeeLake  fp32CoffeeLake  \
0        -0.280223        -0.289843         -0.293034     

## RQ2: SK-ESD descriptive statistics -> `sk_esd_summary_results.csv`


In [10]:
# outputs the CSV for (np)sk-esd descriptive statistic, such as mean with confidence interval and effectsize.

import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, conversion, default_converter
from rpy2.robjects.packages import importr

# 1. Import the R library
sk_esd_pkg = importr('ScottKnottESD')

# Remove objects from previous runs 
robjects.r('rm(list = ls(all.names = TRUE))') 

# Assign your dataframe to an R variable named 'r_df'
with robjects.conversion.localconverter(robjects.default_converter + pandas2ri.converter):
    robjects.r.assign("r_df", skesd_sample)

# Run everything via R strings (Including the CSV writing)
robjects.r('''
    library(ScottKnottESD)
    library(ggplot2)
    library(svglite)
    library(effsize) # Used for calculating distribution-free Cliff's delta
    
    # 1. Run the Scott-Knott ESD test
    sk <- sk_esd(r_df, non.parametric=TRUE)
    
    # --- GGPLOT2 REMAKE SECTION ---
    plot_data <- data.frame(
        Group = names(sk$groups),
        Rank  = as.factor(sk$groups)
    )
    
    raw_long <- stack(r_df)
    names(raw_long) <- c("Value", "Group")
    
    gg_df <- merge(raw_long, plot_data, by="Group")
    
    gg <- ggplot(gg_df, aes(x = reorder(Group, Value, FUN = median), y = Value, fill = Rank)) +
        geom_boxplot(alpha = 0.7, outlier.size = 1, lwd = 0.5) +
        theme_minimal(base_size = 14) +
        scale_fill_viridis_d(option = "plasma") + 
        labs(
            title = "Scott-Knott ESD Grouping Results",
            subtitle = "Groups sharing a color are statistically identical",
            x = "Treatments / Groups",
            y = "Values",
            fill = "Stat Rank"
        ) +
        theme(
            plot.title = element_text(face = "bold", hjust = 0.5),
            plot.subtitle = element_text(hjust = 0.5, color = "darkgray"),
            axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1)
        )
    
    ggsave("twin_skesd_ggplot.png", plot = gg, width = 8, height = 6, dpi = 300)
    ggsave("twin_skesd_plot.pdf", plot = gg, width = 8, height = 6)
    ggsave("twin_skesd_plot.svg", plot = gg, width = 8, height = 6)
    
    # --- DATA EXTRACTION & EXPORT WITHIN R ---
    unique_groups <- names(sk$groups)
    csv_data <- data.frame()
    
    # Global vector pooling all data together to use as a baseline for effect size
    global_values <- raw_long$Value
    
    for(g in unique_groups) {
        group_values <- r_df[[g]]
        group_values <- group_values[!is.na(group_values)] # Clear out structural NAs if unbalanced
        
        grp_rank <- sk$groups[g]
        
        # Calculate Mean and Parametric 95% Confidence Intervals
        grp_mean <- mean(group_values)
        n <- length(group_values)
        se <- sd(group_values) / sqrt(n)
        error <- qt(0.975, df=n-1) * se
        ci_lower <- grp_mean - error
        ci_upper <- grp_mean + error
        ci_string <- sprintf("[%.4f, %.4f]", ci_lower, ci_upper)
        
        # Calculate Effect Size (Cliff's Delta of group vs all pooled data)
        cd_res <- effsize::cliff.delta(group_values, global_values)
        eff_size_val <- sprintf("%.4f (%s)", cd_res$estimate, cd_res$magnitude)
        
        row_df <- data.frame(
            Ranking = grp_rank,
            GroupName = g,
            Mean = round(grp_mean, 5),
            ConfidenceInterval = ci_string,
            CliffsDelta_Global = eff_size_val,
            stringsAsFactors = FALSE
        )
        csv_data <- rbind(csv_data, row_df)
    }
    
    # Sort data cleanly by Statistical Ranking
    csv_data <- csv_data[order(csv_data$Ranking), ]
    
    # Write the CSV directly from R's workspace
    write.csv(csv_data, file = "sk_esd_summary_results.csv", row.names = FALSE)
    
    # --- CAPTURE STATISTICS FOR THE OLD METRIC PLOT ---
    anova_summary <- capture.output(summary(sk$av))
    group_summary <- capture.output(print(sk$groups))

    png("twin_skesd_Stats.png", width=700, height=500)
    plot.new()
    full_output <- c("--- ANOVA Table ---", anova_summary, "", "--- Scott-Knott Grouping ---", group_summary)
    text(x=0, y=1, labels=paste(full_output, collapse="\n"), adj=c(0,1), family="mono", cex=0.9)
    dev.off()     
''')

print("[SUCCESS] Execution complete. R has generated and saved 'sk_esd_summary_results.csv' locally.")

[SUCCESS] Execution complete. R has generated and saved 'sk_esd_summary_results.csv' locally.


## RQ2: Figure `fig:skesd` (per-CPU ranking plots)


In [2]:
# Cell #4: Generate TWO SEPARATE SK-ESD plots, one for Coffee Lake and one for PC2 (Arrow Lake).
# Each CPU's data is run through its own non-parametric Scott-Knott ESD test and plotted
# independently, reusing the ggplot2 boxplot styling from cell #3.

import pandas as pd
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri, conversion, default_converter
from rpy2.robjects.packages import importr

# 1. Import the R library
sk_esd_pkg = importr('ScottKnottESD')

# Remove objects from previous runs
robjects.r('rm(list = ls(all.names = TRUE))')

# Assign each CPU's dataframe to its own R variable
with robjects.conversion.localconverter(robjects.default_converter + pandas2ri.converter):
    robjects.r.assign("r_df_coffee", skesd_sample_coffee)
    robjects.r.assign("r_df_pc2", skesd_sample_pc2_trimmed)

# Run a separate SK-ESD + ggplot2 for each CPU using a reusable R function
robjects.r('''
    library(ScottKnottESD)
    library(ggplot2)
    library(svglite)

    plot_skesd <- function(df, title_str, file_prefix) {
        # 1. Run the Scott-Knott ESD test on this CPU's data
        sk <- sk_esd(df, non.parametric=TRUE)

        # Extract ranking results
        plot_data <- data.frame(
            Group = names(sk$groups),
            Rank  = as.factor(sk$groups)
        )

        # Reshape raw data into long format and attach the SK ranks
        raw_long <- stack(df)
        names(raw_long) <- c("Value", "Group")
        gg_df <- merge(raw_long, plot_data, by="Group")

        # Build the ggplot (same style as cell #3)
        gg <- ggplot(gg_df, aes(x = reorder(Group, Value, FUN = median), y = Value, fill = Rank)) +
            geom_boxplot(alpha = 0.7, outlier.size = 1, lwd = 0.5) +
            theme_minimal(base_size = 14) +
            scale_fill_viridis_d(option = "plasma") +
            labs(
                title = title_str,
                subtitle = "Groups sharing a color are statistically identical",
                x = "Treatments / Groups",
                y = "Values",
                fill = "Stat Rank"
            ) +
            theme(
                plot.title = element_text(face = "bold", hjust = 0.5),
                plot.subtitle = element_text(hjust = 0.5, color = "darkgray"),
                axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1)
            )

        ggsave(paste0(file_prefix, "_ggplot.png"), plot = gg, width = 8, height = 6, dpi = 300)
        ggsave(paste0(file_prefix, "_plot.pdf"),  plot = gg, width = 8, height = 6)
        ggsave(paste0(file_prefix, "_plot.svg"),  plot = gg, width = 8, height = 6)

        return(gg)
    }

    # Coffee Lake plot
    plot_skesd(r_df_coffee, "Scott-Knott ESD Grouping Results - Coffee Lake", "skesd_coffee")

    # PC2 / Arrow Lake plot
    plot_skesd(r_df_pc2, "Scott-Knott ESD Grouping Results - PC2 (Arrow Lake)", "skesd_pc2")
''')

print("[SUCCESS] Generated two separate SK-ESD plots:")
print("  Coffee Lake -> skesd_coffee_ggplot.png / .pdf / .svg")
print("  PC2 (Arrow Lake) -> skesd_pc2_ggplot.png / .pdf / .svg")


Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.
R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: package 'ggplot2' was built under R version 4.4.3 
  
R callback write-console: 2: package 'svglite' was built under R version 4.4.3 
  


[SUCCESS] Generated two separate SK-ESD plots:
  Coffee Lake -> skesd_coffee_ggplot.png / .pdf / .svg
  PC2 (Arrow Lake) -> skesd_pc2_ggplot.png / .pdf / .svg


### LaTeX body for Table `tableskesd`


In [5]:
df_skesd_result = pd.read_csv("./sk_esd_summary_results.csv")
print(df_skesd_result.head(1))
print(df_skesd_result.to_latex())


   Ranking        GroupName     Mean  ConfidenceInterval CliffsDelta_Global
0        1  fp64_Arrow_Lake -0.08578  [-0.0917, -0.0799]     0.8736 (large)
\begin{tabular}{lrlrll}
\toprule
 & Ranking & GroupName & Mean & ConfidenceInterval & CliffsDelta_Global \\
\midrule
0 & 1 & fp64_Arrow_Lake & -0.085780 & [-0.0917, -0.0799] & 0.8736 (large) \\
1 & 2 & MMX_Arrow_Lake & -0.099700 & [-0.1034, -0.0960] & 0.7358 (large) \\
2 & 2 & fp32_Arrow_Lake & -0.099870 & [-0.1043, -0.0955] & 0.7173 (large) \\
3 & 3 & SSE_Arrow_Lake & -0.106420 & [-0.1102, -0.1027] & 0.6007 (large) \\
4 & 3 & AVX_Arrow_Lake & -0.108330 & [-0.1130, -0.1037] & 0.5195 (large) \\
5 & 4 & SSE2_Arrow_Lake & -0.110380 & [-0.1157, -0.1051] & 0.4585 (medium) \\
6 & 4 & AVX2_Arrow_Lake & -0.113550 & [-0.1183, -0.1088] & 0.4281 (medium) \\
7 & 5 & int32_Arrow_Lake & -0.121160 & [-0.1265, -0.1158] & 0.2552 (small) \\
8 & 5 & int64_Arrow_Lake & -0.123440 & [-0.1289, -0.1180] & 0.2225 (small) \\
9 & 5 & int128_Arrow_Lake & -0.125500